In [22]:
import pandas as pd
import glob 
import numpy as np
import protfasta
from ete4 import Tree
from ete4.treeview import TreeStyle, TextFace, NodeStyle
import re
from pandarallel import pandarallel


In [25]:
plant_yeast_mammal_dict = protfasta.read_fasta("../output/plant_yeast_mammal_filtered.fasta")
plant_yeast_mammal_df = pd.DataFrame({'id': plant_yeast_mammal_dict.keys(), 'seq': plant_yeast_mammal_dict.values()})
plant_yeast_mammal_df

,id,seq
0,g001536.m1_brettanomyces_anomalus.final,MSDKPHNQYACTRCRRLKKKCSKEFPRCANCRKQNKVCEYVERKNKRKRPLPGSGANKGGSVLNGDLYSGNNGNNGNNNNDNINNNNNNDNFNFGN...
1,C1_01200W_A_candida_albicans.cgd,MRPIDSIVSKFRVQKHTSNLADNPAVLKHPEDMPKLTSCLRCRKLKKKCDKSTPHCLNCE
2,g002356.m1_candida_orthopsilosis.final,MLSPDSSVTKFRVKKFPVTDGNGNNFSHHQIEQISHEPVAKVTSCLRCRKLKKKCNKAKPECSNCDKAGEKCQYVPRKQRVTKKDKLKLKLEGPTT...
3,g001338.m1_candida_parapsilosis.final,MLSPESSVTKFRVKKFPVTDGNGNNFSHHQIEQIANEPVAKVTSCLRCRKLKKKCNKAKPECSNCDKAGEECQYVPRKQRVTKRDKLKFNGDGPST...
4,g002226.m1_candida_sojae.final,MSTEPIVTKFRVKKPTTKAPSSNDLIKDISKLTSCLRCRKLKKKCNKNTPECLNCDKAGEECTYVPRKQRRVKNDLKDLPLNAEDHIGSSQPTSPD...
...,...,...
405265,tr|W8BNV6|W8BNV6_CERCA Aminoacyl tRNA synthase complex-interacting multifunctional protein 1 OS=...,MIIWSLLVIHNSLTMSLNRILAFGAGRIPISLLQRQTQQLKMPVLEEIIAKNKRNQDIITSIASELKVMRGELLARKKQQLSQENALLRQQVEVAK...
405266,tr|W8C4M4|W8C4M4_CERCA (Mediterranean fruit fly) hypothetical protein OS=Ceratitis capitata OX=7...,MYNGECGILIKDIKPGLKNINVIFIVLEIGTATVTKENREVRNFKVGDHSACINVSIWDEPGKLIAPGDIIKLTKGYASIWRHCLTLYSGKNGEVY...
405267,tr|W8C822|W8C822_CERCA (Mediterranean fruit fly) hypothetical protein OS=Ceratitis capitata OX=7...,MSQLNTPEKDAAGKPPTSRQQSTHSPNLTLQLPSPIITKRTRTASTSARALENPIETGIVKSFSRSKGHGFITPSAGGEDLFCHVSDVDGEYVPQQ...
405268,tr|W8QUS6|W8QUS6_MIMNO Cold shock domain protein OS=Mimachlamys nobilis OX=106276 PE=2 SV=1,MSDSEKQPEEKEEPKKNVIATKVTGTVKWFNVKSGYGFINRDDTKEDVFVHQTAITKNNPRKYLRSVGDGEKVEFDVVEGEKGNEASNVTGPEGNP...


In [ ]:
plant_yeast_mammal_df.sample(10)["id"]

303312                                              Vang0441s00120.1 Vigna angularis|C2H2|C2H2 family protein
113084                                                  XP_009134642.1 Brassica rapa|C2H2|C2H2 family protein
249231                                                    XP_008228998.1 Prunus mume|bHLH|bHLH family protein
131940                                      Cagra.9228s0001.1.p Capsella grandiflora|bHLH|bHLH family protein
383498                sp|Q32KS7|ZN821_BOVIN Zinc finger protein 821 OS=Bos taurus OX=9913 GN=ZNF821 PE=2 SV=1
105218                                          GSBRNA2T00137180001 Brassica napus|NF-YA|NF-YA family protein
316732                                    Zpz_sc02481.1.g00110.1.sm.mk Zoysia pacifica|GRF|GRF family protein
353079    tr|A0AAA9U1G2|A0AAA9U1G2_BOVIN Zinc finger and BTB domain containing 40 OS=Bos taurus OX=9913 GN...
107193                                            GSBRNA2T00121167001 Brassica napus|GATA|GATA family protein
142671    

In [48]:

def extract_species(x):
    # 1. UniProt style: OS=Species ... OX=
    m = re.search(r'OS=([^=]+?)\s+OX=', x)
    if m:
        species = m.group(1).strip()
        species = re.sub(r'\(strain.*?\)', '', species).strip()
        return species

    # 2. Species before '|' (multi-word, may include numbers)
    m = re.search(r'\s([A-Z][a-z]+(?:\s[a-zA-Z0-9\.-]+){1,3})\|', x)
    if m:
        return m.group(1).strip()

    # 3. Species at end of string (no '|')
    m = re.search(r'\s([A-Z][a-z]+(?:\s[a-zA-Z0-9\.-]+){1,3})$', x)
    if m:
        return m.group(1).strip()

    # 4. _genus_species lowercase
    m = re.search(r'_([a-z]+)_([a-z0-9]+)', x)
    if m:
        return f"{m.group(1).capitalize()} {m.group(2)}"

    # 5. _Genus_species capitalized
    m = re.search(r'_([A-Z][a-z]+)_([a-z0-9]+)', x)
    if m:
        return f"{m.group(1)} {m.group(2)}"

    # 6. _Genus_sp placeholder
    m = re.search(r'_([A-Z][a-z]+)_sp', x)
    if m:
        return f"{m.group(1)} sp."

    return None


In [49]:

# Initialize pandarallel (auto-detects cores)
pandarallel.initialize(progress_bar=False)

# Parallel apply
plant_yeast_mammal_df["species"] = plant_yeast_mammal_df["id"].parallel_apply(extract_species)
plant_yeast_mammal_df

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


,id,seq,species
0,g001536.m1_brettanomyces_anomalus.final,MSDKPHNQYACTRCRRLKKKCSKEFPRCANCRKQNKVCEYVERKNKRKRPLPGSGANKGGSVLNGDLYSGNNGNNGNNNNDNINNNNNNDNFNFGN...,Brettanomyces anomalus
1,C1_01200W_A_candida_albicans.cgd,MRPIDSIVSKFRVQKHTSNLADNPAVLKHPEDMPKLTSCLRCRKLKKKCDKSTPHCLNCE,Candida albicans
2,g002356.m1_candida_orthopsilosis.final,MLSPDSSVTKFRVKKFPVTDGNGNNFSHHQIEQISHEPVAKVTSCLRCRKLKKKCNKAKPECSNCDKAGEKCQYVPRKQRVTKKDKLKLKLEGPTT...,Candida orthopsilosis
3,g001338.m1_candida_parapsilosis.final,MLSPESSVTKFRVKKFPVTDGNGNNFSHHQIEQIANEPVAKVTSCLRCRKLKKKCNKAKPECSNCDKAGEECQYVPRKQRVTKRDKLKFNGDGPST...,Candida parapsilosis
4,g002226.m1_candida_sojae.final,MSTEPIVTKFRVKKPTTKAPSSNDLIKDISKLTSCLRCRKLKKKCNKNTPECLNCDKAGEECTYVPRKQRRVKNDLKDLPLNAEDHIGSSQPTSPD...,Candida sojae
...,...,...,...
405265,tr|W8BNV6|W8BNV6_CERCA Aminoacyl tRNA synthase complex-interacting multifunctional protein 1 OS=...,MIIWSLLVIHNSLTMSLNRILAFGAGRIPISLLQRQTQQLKMPVLEEIIAKNKRNQDIITSIASELKVMRGELLARKKQQLSQENALLRQQVEVAK...,Ceratitis capitata
405266,tr|W8C4M4|W8C4M4_CERCA (Mediterranean fruit fly) hypothetical protein OS=Ceratitis capitata OX=7...,MYNGECGILIKDIKPGLKNINVIFIVLEIGTATVTKENREVRNFKVGDHSACINVSIWDEPGKLIAPGDIIKLTKGYASIWRHCLTLYSGKNGEVY...,Ceratitis capitata
405267,tr|W8C822|W8C822_CERCA (Mediterranean fruit fly) hypothetical protein OS=Ceratitis capitata OX=7...,MSQLNTPEKDAAGKPPTSRQQSTHSPNLTLQLPSPIITKRTRTASTSARALENPIETGIVKSFSRSKGHGFITPSAGGEDLFCHVSDVDGEYVPQQ...,Ceratitis capitata
405268,tr|W8QUS6|W8QUS6_MIMNO Cold shock domain protein OS=Mimachlamys nobilis OX=106276 PE=2 SV=1,MSDSEKQPEEKEEPKKNVIATKVTGTVKWFNVKSGYGFINRDDTKEDVFVHQTAITKNNPRKYLRSVGDGEKVEFDVVEGEKGNEASNVTGPEGNP...,Mimachlamys nobilis


In [50]:
plant_yeast_mammal_df[plant_yeast_mammal_df["species"].isna()]

,id,seq,species
745,g002677.m1_yHKB357_New_Genus_SPADES.final,MNGMSHPQSSQQQQQQPITLLQDAWQRSYPGADRQRAIQYLVAALREVHGSDFDLQRSTIMASEFEKFTYVKSLTKEDYVQAMRLKITQVHSSSRP...,None
3003,g001536.m1_yHKB357_New_Genus_SPADES.final,MSEIDTFFDFNDFIGSIAPSAASPASIHSDTSPHPQSLFFGLAQHDSAAAPVPATDYAEPSGSDSPRDLSLSNSSVPTSPDIVKTEPASFSNSPTF...,None
3004,g006726.m1_yHKB357_New_Genus_SPADES.final,MGFDKVSVATASRDSPKWSHEYTEEPRLPTQLASLTLPQYLPFNGTYVASNKFDHTSSALEYTLERLQTSSTRSNSTDSPLTSQFPKRIEPAYHTS...,None
5872,g003144.m1_yHKB357_New_Genus_SPADES.final,MALASSSALMSPRQIPSQHSKHSGAGARNRPRVVRACDCCRDHKLRCNADFDSLLNIVVAPCDRCVKHQSVCSFARTPLKRGPRAGFRRKHSLHRT...,None
9065,g003858.m1_yHKB357_New_Genus_SPADES.final,MLSEANPMSSSSGPVFDSQDFMGHLPSKAESAFEASPTMFDSPFGAEDDLASPLLINPSVLDSVFSSVLDDHSGVQDHTPMFDDLELGPESEWGLL...,None
...,...,...,...
176181,KDD73540.1 Helicosporidium|AP2|AP2 family protein,MEVPACTDVGALGVSGAVAPVKRRRSGPKSKSSPYIGVTQYKRTGRYEAHIWIVEMRGAKGHQRHLGSYATAEDAARCYDRAALRLRGPGAELNFP...,None
319119,tr|A0A0R1G188|A0A0R1G188_9LACO Insertion element IS150 protein InsJ-like helix-turn-helix domain...,MFSKELKLAAIKDYYNGLGFTKITNKYGIKGSATLYEWISKVEQFGIEYFSKNTKTYYEYSFKIKVINWRLKNHESLTNAARHFAISAPFVIYQWE...,None
321580,tr|A0A218P4W7|A0A218P4W7_THECE Small ribosomal subunit protein eS19 OS=Thermococcus celer Vu 13 ...,MATVYDVPGDLLVERTAKALKEVEAIKPPEWAPFVKTGRHKERIPEQEDWWYYRVASIFRKIYIDGPVGIERLRTWYGGRKNRGHAPEHFYKAGGS...,None
356825,tr|A0A218NZX3|A0A218NZX3_THECE Small ribosomal subunit protein uS17 OS=Thermococcus celer Vu 13 ...,MREIGLKVQPPAEKCDDPHCPWHGHLRIHGRYFEGIVVSDKGKKTVVVERRHYHYLKKYERYELRRSKVHAHNPECIDAKVGDRVLVAETRPISKT...,None


In [56]:
plant_yeast_mammal_df[plant_yeast_mammal_df["species"].isna()].sample(10)["id"]

31179             g006667.m1_yHKB357_New_Genus_SPADES.final
13965             g004984.m1_yHKB357_New_Genus_SPADES.final
176161    KDD76345.1 Helicosporidium|SBP|SBP family protein
70666             g005018.m1_yHKB357_New_Genus_SPADES.final
55625             g005966.m1_yHKB357_New_Genus_SPADES.final
23201             g000233.m1_yHKB357_New_Genus_SPADES.final
21009             g006004.m1_yHKB357_New_Genus_SPADES.final
60138             g004587.m1_yHKB357_New_Genus_SPADES.final
176173    KDD72353.1 Helicosporidium|MYB|MYB family protein
31178             g003738.m1_yHKB357_New_Genus_SPADES.final
Name: id, dtype: object

In [61]:
# Make sure the species looks good
plant_yeast_mammal_df[~plant_yeast_mammal_df["species"].isna()].sample(10)[["id", "species"]]

,id,species
197871,MDP0000144744 Malus domestica|MYB|MYB family protein,Malus domestica
359163,sp|P23757|POXM_DROME Paired box pox-meso protein OS=Drosophila melanogaster OX=7227 GN=Poxm PE=2...,Drosophila melanogaster
352069,tr|A0A8N5EMC9|A0A8N5EMC9_GEOFO Zinc finger CCCH domain-containing protein 7B isoform X4 OS=Geosp...,Geospiza fortis
392523,tr|A0A067ZVF2|A0A067ZVF2_BRAOV Dehydration-responsive element-binding protein 2A (Fragment) OS=B...,Brassica oleracea var. viridis
170594,Glyma.20G005800.1.p Glycine max|C2H2|C2H2 family protein,Glycine max
252317,Prupe.1G173300.1.p Prunus persica|bHLH|bHLH family protein,Prunus persica
64838,g002761.m1_yHMPu5000034962_hanseniaspora_guilliermondii_170307.final,Hanseniaspora guilliermondii
88683,Aqcoe3G081500.1.p Aquilegia coerulea|TCP|TCP family protein,Aquilegia coerulea
156283,462924531 Eragrostis tef|NAC|NAC family protein,Eragrostis tef
382576,tr|M4M668|M4M668_HELVI GATA zinc finger domain containing protein 1 OS=Heliothis virescens OX=71...,Heliothis virescens


In [66]:
plant_yeast_mammal_df["species"] = plant_yeast_mammal_df["species"].str.split(r" \(").str[0]
plant_yeast_mammal_df["species"] = plant_yeast_mammal_df["species"].str.split(" serotype").str[0]
plant_yeast_mammal_df["species"] = plant_yeast_mammal_df["species"].str.split(" serogroup").str[0]
plant_yeast_mammal_df["species"] = plant_yeast_mammal_df["species"].str.split("/").str[0]
plant_yeast_mammal_df["species"] = plant_yeast_mammal_df["species"].str.split(" serovar").str[0]
plant_yeast_mammal_df["species"] = plant_yeast_mammal_df["species"].str.split(r" sp$").str[0]
plant_yeast_mammal_df

,id,seq,species
0,g001536.m1_brettanomyces_anomalus.final,MSDKPHNQYACTRCRRLKKKCSKEFPRCANCRKQNKVCEYVERKNKRKRPLPGSGANKGGSVLNGDLYSGNNGNNGNNNNDNINNNNNNDNFNFGN...,Brettanomyces anomalus
1,C1_01200W_A_candida_albicans.cgd,MRPIDSIVSKFRVQKHTSNLADNPAVLKHPEDMPKLTSCLRCRKLKKKCDKSTPHCLNCE,Candida albicans
2,g002356.m1_candida_orthopsilosis.final,MLSPDSSVTKFRVKKFPVTDGNGNNFSHHQIEQISHEPVAKVTSCLRCRKLKKKCNKAKPECSNCDKAGEKCQYVPRKQRVTKKDKLKLKLEGPTT...,Candida orthopsilosis
3,g001338.m1_candida_parapsilosis.final,MLSPESSVTKFRVKKFPVTDGNGNNFSHHQIEQIANEPVAKVTSCLRCRKLKKKCNKAKPECSNCDKAGEECQYVPRKQRVTKRDKLKFNGDGPST...,Candida parapsilosis
4,g002226.m1_candida_sojae.final,MSTEPIVTKFRVKKPTTKAPSSNDLIKDISKLTSCLRCRKLKKKCNKNTPECLNCDKAGEECTYVPRKQRRVKNDLKDLPLNAEDHIGSSQPTSPD...,Candida sojae
...,...,...,...
405265,tr|W8BNV6|W8BNV6_CERCA Aminoacyl tRNA synthase complex-interacting multifunctional protein 1 OS=...,MIIWSLLVIHNSLTMSLNRILAFGAGRIPISLLQRQTQQLKMPVLEEIIAKNKRNQDIITSIASELKVMRGELLARKKQQLSQENALLRQQVEVAK...,Ceratitis capitata
405266,tr|W8C4M4|W8C4M4_CERCA (Mediterranean fruit fly) hypothetical protein OS=Ceratitis capitata OX=7...,MYNGECGILIKDIKPGLKNINVIFIVLEIGTATVTKENREVRNFKVGDHSACINVSIWDEPGKLIAPGDIIKLTKGYASIWRHCLTLYSGKNGEVY...,Ceratitis capitata
405267,tr|W8C822|W8C822_CERCA (Mediterranean fruit fly) hypothetical protein OS=Ceratitis capitata OX=7...,MSQLNTPEKDAAGKPPTSRQQSTHSPNLTLQLPSPIITKRTRTASTSARALENPIETGIVKSFSRSKGHGFITPSAGGEDLFCHVSDVDGEYVPQQ...,Ceratitis capitata
405268,tr|W8QUS6|W8QUS6_MIMNO Cold shock domain protein OS=Mimachlamys nobilis OX=106276 PE=2 SV=1,MSDSEKQPEEKEEPKKNVIATKVTGTVKWFNVKSGYGFINRDDTKEDVFVHQTAITKNNPRKYLRSVGDGEKVEFDVVEGEKGNEASNVTGPEGNP...,Mimachlamys nobilis


In [67]:
plant_yeast_mammal_df[["species"]].drop_duplicates().to_csv("../data/expanded_input_data_species.txt", index=False, header=False)

In [49]:
cl_seqs["species"].iloc[-1]

'g005029.m1_yHRVM31_Schwanniomyces_sp_nov_plate33_SPADES'

In [33]:
cl_seqs[cl_seqs["species"].str.contains(r"\d")]

,id,seq,species
0,g003334.m1_alloascoidea_hylecoeti.final,MEQQPTQDNISNETHGSEEPPTKKSRSDNGTECETTDATPTINRNE...,g003334.m1_alloascoidea_hylecoeti
1,g002909.m1_arxula_adeninivorans.final,MAEPDQISEGHESPGSSNKRQLHEVDHSDRSPRPEDTNGTKRPRTD...,g002909.m1_arxula_adeninivorans
2,g001599.m1_ashbya_aceri.final,MALGIPYLHEYDEDTLKQRLEAVLPGLPYATQLTLKSLPLLNDIST...,g001599.m1_ashbya_aceri
3,g002398.m1_brettanomyces_anomalus.final,MADPDTSHKSPLLSENNLPHNNKKISDNVPTEEDGKKEAQTETPQI...,g002398.m1_brettanomyces_anomalus
4,C2_04200W_A_candida_albicans.cgd,MSDQLEKDIEESIANLDYQQNQEHHETEQDKDKEHQDVEKQSSEEE...,C2_04200W_A_candida_albicans
...,...,...,...
110807,g005054.m1_saccharomyces_mikatae.final,MMDMQVRKVRKPPACTQCRKRKIGCDRAKPICGNCVKYNKPDCFYP...,g005054.m1_saccharomyces_mikatae
110808,g005100.m1_saccharomyces_paradoxus.final,MMDMQVRKVRKPPACTQCRKRKIGCDRAKPICGNCVKYNKPDCFYP...,g005100.m1_saccharomyces_paradoxus
110809,g002076.m1_saccharomyces_uvarum.final,MMDMQVRKVRKPPACTQCRKRKIGCDRAKPICGNCVKYNKPDCFYP...,g002076.m1_saccharomyces_uvarum
110810,g004724.m1_yHMPu5000034597_candida_stellimalic...,MNSNGSFKSIKGFNKVLNLIKYMLRFSKKKQNFARNSDNNNVTDYS...,g004724.m1_yHMPu5000034597_candida_stellimalic...


In [13]:
cl_seqs[cl_seqs["species"].str.contains(r"\d")]

ValueError: Cannot mask with non-boolean array containing NA / NaN values